# Equity Curve Trailing Stop Strategy on SPY
## Strategy Brief
The Equity Curve Trailing Stop Strategy aims to capitalize on upward trends in the SPY ETF by implementing a trailing stop based on the equity curve. The strategy generates buy signals when the equity curve is above a specified moving average and sells when it falls below a trailing stop level. This approach seeks to maximize gains during bullish periods while minimizing losses during downturns. Historical backtesting will be conducted to evaluate the strategy's performance against a buy-and-hold approach.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

### PHASE 1 - Trading Context
In this phase, we define the parameters for our strategy, including the moving average window for the equity curve and the trailing stop percentage.

In [ ]:
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'
EQUITY_MA_WINDOW = 50
TRAILING_STOP_PERCENT = 0.1

### PHASE 2 - Data Exploration
We will download historical price data for SPY using yfinance, calculate the equity curve, and visualize it alongside the SPY price to understand the relationship.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download('SPY', start=START_DATE, end=END_DATE)
data['Returns'] = data['Adj Close'].pct_change()

# Calculate equity curve
initial_investment = 10000
equity_curve = (1 + data['Returns']).cumprod() * initial_investment

# Plot
plt.figure(figsize=(14, 7))
plt.plot(data.index, data['Adj Close'], label='SPY Price')
plt.plot(data.index, equity_curve, label='Equity Curve', linestyle='--')
plt.title('SPY Price and Equity Curve')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.show()

### PHASE 3 - Strategy Engineering
We will create a signal series based on the equity curve's moving average and apply the trailing stop logic to determine entry and exit points.

In [ ]:
# Calculate moving average of equity curve
equity_ma = equity_curve.rolling(window=EQUITY_MA_WINDOW).mean()

# Generate signals
signals = pd.Series(index=data.index)
signals[equity_curve > equity_ma] = 1
signals[equity_curve <= equity_ma] = 0

# Apply trailing stop logic
trailing_stop = equity_curve.rolling(window=EQUITY_MA_WINDOW).max() * (1 - TRAILING_STOP_PERCENT)
positions = (equity_curve > trailing_stop).astype(int)

### PHASE 4 - Coding & Backtesting
Shift positions by one day to avoid look-ahead bias, calculate daily returns based on positions, and plot the resulting equity curve.

In [ ]:
# Shift positions
positions = positions.shift(1).fillna(0)

# Calculate strategy returns
strategy_returns = data['Returns'] * positions
strategy_equity_curve = (1 + strategy_returns).cumprod() * initial_investment

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(data.index, strategy_equity_curve, label='Strategy Equity Curve')
plt.title('Strategy Equity Curve')
plt.xlabel('Date')
plt.ylabel('Equity')
plt.legend()
plt.show()

### PHASE 5 - Performance Evaluation
Evaluate the strategy's performance using key metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown, and compare it to a buy-and-hold strategy.

In [ ]:
def calculate_performance_metrics(equity_curve):
    # Calculate CAGR
    total_return = equity_curve[-1] / equity_curve[0]
    num_years = (equity_curve.index[-1] - equity_curve.index[0]).days / 365.25
    cagr = (total_return ** (1 / num_years)) - 1

    # Calculate Sharpe ratio
    daily_returns = equity_curve.pct_change().dropna()
    sharpe_ratio = np.mean(daily_returns) / np.std(daily_returns) * np.sqrt(252)

    # Calculate Sortino ratio
    downside_returns = daily_returns[daily_returns < 0]
    sortino_ratio = np.mean(daily_returns) / np.std(downside_returns) * np.sqrt(252)

    # Calculate Calmar ratio
    max_drawdown = (equity_curve / equity_curve.cummax() - 1).min()
    calmar_ratio = cagr / abs(max_drawdown)

    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

# Strategy performance
strategy_metrics = calculate_performance_metrics(strategy_equity_curve)

# Buy-and-hold performance
buy_and_hold_metrics = calculate_performance_metrics(equity_curve)

# Comparison table
comparison_df = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'Strategy': strategy_metrics,
    'Buy and Hold': buy_and_hold_metrics
})

comparison_df

### PHASE 6 - Deploy & Monitor
Create a function to download the last 60 days of SPY data, compute today's signal, and print the recommended position.

In [ ]:
def get_latest_signal():
    # Download last 60 days of data
    recent_data = yf.download('SPY', period='60d')
    recent_data['Returns'] = recent_data['Adj Close'].pct_change()
    
    # Calculate recent equity curve
    recent_equity_curve = (1 + recent_data['Returns']).cumprod() * initial_investment
    
    # Calculate moving average and trailing stop
    recent_equity_ma = recent_equity_curve.rolling(window=EQUITY_MA_WINDOW).mean()
    recent_trailing_stop = recent_equity_curve.rolling(window=EQUITY_MA_WINDOW).max() * (1 - TRAILING_STOP_PERCENT)
    
    # Determine signal
    if recent_equity_curve.iloc[-1] > recent_trailing_stop.iloc[-1]:
        position = 'Long'
    else:
        position = 'Cash'
    
    print(f"Today's position: {position}")

get_latest_signal()